In [1]:
"""
ALGORITHM 1: Privacy Auditor with One Training Run
From: "Privacy Auditing with One (1) Training Run" (Steinke, Nasr, Jagielski, 2023)

This is a standalone implementation of Algorithm 1 from the paper.
The algorithm audits differential privacy by randomly including/excluding canaries
in a single training run and using membership scores to infer ε.
"""

import numpy as np
import torch
import torch.nn as nn
from scipy import stats
from typing import Tuple, Dict, Any, Callable
from pathlib import Path

# ============================================================================
# ALGORITHM 1 IMPLEMENTATION
# ============================================================================

class PrivacyAuditor:
    """
    Implements Algorithm 1: Auditor with One Training Run
    
    Key components:
    1. Random canary selection (m examples)
    2. Bernoulli(0.5) inclusion/exclusion per canary
    3. Score computation: Score(x, w) = loss(w_0, x) - loss(w_T, x)
    4. Statistical test via Theorem 5.2 to get ε lower bound
    """
    
    def __init__(
        self,
        m: int,           # Number of auditing examples (canaries)
        k_plus: int,      # Number of positive guesses
        k_minus: int,     # Number of negative guesses
        n_total: int,     # Total dataset size (m + non-auditing)
    ):
        self.m = m
        self.k_plus = k_plus
        self.k_minus = k_minus
        self.n_total = n_total
        self.n_non_auditing = n_total - m
        
        # Will be populated during execution
        self.S = None          # True inclusion/exclusion labels
        self.T = None          # Auditor's guesses
        self.scores = None     # Membership scores
        self.w_0 = None        # Initial weights
        self.w_T = None        # Final weights
    
    # ─── STEP 1: Generate Random Inclusion/Exclusion ───────────────────────
    def generate_random_selection(self, seed: int = 42) -> np.ndarray:
        """
        Algorithm 1, Line 3: For i ∈ [m] sample S_i ∈ {-1, +1} independently
        with E[S_i] = 0. Set S_i = 1 for all i ∈ [n] \ [m].
        
        Returns:
            S: np.ndarray of shape (n,) with values {-1, +1}
               S[i] = +1 means example i is IN
               S[i] = -1 means example i is OUT
        """
        np.random.seed(seed)
        
        # Initialize all to +1 (always included)
        S = np.ones(self.n_total, dtype=np.int32)
        
        # For first m examples, flip fair coin: -1 or +1
        S[:self.m] = np.random.choice([-1, +1], size=self.m, p=[0.5, 0.5])
        
        self.S = S
        return S
        # Returns binary inclusion vector: +1=IN, -1=OUT
        # First m entries randomized, rest always +1
    
    # ─── STEP 2: Partition Dataset by Selection ────────────────────────────
    def partition_dataset(
        self, 
        X: np.ndarray, 
        y: np.ndarray
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        """
        Algorithm 1, Line 4: Partition x into x_IN and x_OUT according to S.
        
        Args:
            X: All examples (n_total, ...)
            y: All labels (n_total,)
        
        Returns:
            X_IN, y_IN, X_OUT, y_OUT
        """
        assert self.S is not None, "Must call generate_random_selection() first"
        
        # Split based on S
        mask_IN = (self.S == 1)
        mask_OUT = (self.S == -1)
        
        X_IN = X[mask_IN]
        y_IN = y[mask_IN]
        X_OUT = X[mask_OUT]  # Not used in training, only for scoring
        y_OUT = y[mask_OUT]
        
        n_IN = X_IN.shape[0]
        n_OUT = X_OUT.shape[0]
        
        print(f"Dataset partitioned: {n_IN} IN, {n_OUT} OUT")
        
        return X_IN, y_IN, X_OUT, y_OUT
        # Splits dataset into IN (train on) and OUT (exclude)
        # Returns separate arrays for included/excluded examples
    
    # ─── STEP 3: Train Algorithm A ──────────────────────────────────────────
    def run_training_algorithm(
        self,
        algorithm_A: Callable,  # The DP algorithm to audit (e.g., DP-SGD)
        X_IN: np.ndarray,
        y_IN: np.ndarray,
        **train_kwargs
    ) -> Tuple[Any, Any]:
        """
        Algorithm 1, Line 5: Run A on input x_IN with appropriate parameters,
        outputting w.
        
        Args:
            algorithm_A: Training function with signature:
                         (X, y, **kwargs) -> (w_0, w_T)
            X_IN: Training examples
            y_IN: Training labels
            **train_kwargs: Additional parameters for algorithm_A
        
        Returns:
            w_0: Initial model weights (before any training)
            w_T: Final model weights (after T iterations)
        """
        print("Running Algorithm A (the DP algorithm to audit)...")
        
        # Train and get both initial and final weights
        w_0, w_T = algorithm_A(X_IN, y_IN, **train_kwargs)
        
        self.w_0 = w_0
        self.w_T = w_T
        
        return w_0, w_T
        # Executes the DP algorithm being audited
        # Must return BOTH w_0 (initial) and w_T (final) weights
    
    # ─── STEP 4: Compute Membership Scores ─────────────────────────────────
    def compute_scores(
        self,
        X: np.ndarray,
        y: np.ndarray,
        loss_fn: Callable,  # loss_fn(model, x, y) -> scalar loss
        model_w0: Any,      # Model with weights w_0
        model_wT: Any,      # Model with weights w_T
    ) -> np.ndarray:
        """
        Algorithm 1, Line 6: Compute the vector of scores
        Y = (Score(x_i, w) : i ∈ [m])
        
        Score formula (from paper):
            Score(x, w) = loss(w_0, x) - loss(w_T, x)
        
        Intuition: If x was IN training, loss(w_T, x) should be lower,
                   making Score(x, w) positive (high score).
        
        Args:
            X: Examples to score (only first m)
            y: Labels
            loss_fn: Loss function
            model_w0: Model with initial weights
            model_wT: Model with final weights
        
        Returns:
            scores: np.ndarray of shape (m,)
        """
        assert self.w_0 is not None, "Must run training first"
        
        # Only score the first m examples (canaries)
        X_canaries = X[:self.m]
        y_canaries = y[:self.m]
        
        scores = np.zeros(self.m)
        
        model_w0.eval()
        model_wT.eval()
        
        with torch.no_grad():
            for i in range(self.m):
                x_i = torch.tensor(X_canaries[i:i+1], dtype=torch.float32)
                y_i = torch.tensor(y_canaries[i:i+1], dtype=torch.long)
                
                # Loss at initial weights
                loss_w0 = loss_fn(model_w0, x_i, y_i).item()
                
                # Loss at final weights
                loss_wT = loss_fn(model_wT, x_i, y_i).item()
                
                # Score = loss decreased during training
                scores[i] = loss_w0 - loss_wT
        
        self.scores = scores
        
        print(f"Scores computed. Mean: {scores.mean():.4f}, "
              f"Std: {scores.std():.4f}")
        
        return scores
        # Computes membership score for each canary
        # High score → likely IN, Low score → likely OUT
        # Score = loss_initial - loss_final (positive if training helped)
    
    # ─── STEP 5: Make Guesses Based on Scores ──────────────────────────────
    def make_guesses(self, scores: np.ndarray) -> np.ndarray:
        """
        Algorithm 1, Lines 7-8: Sort the scores Y. Let T ∈ {-1, 0, +1}^m be
        +1 for the largest k+ scores and -1 for the smallest k- scores.
        
        Strategy:
        - Top k_plus scores → guess +1 (IN)
        - Bottom k_minus scores → guess -1 (OUT)
        - Middle scores → guess 0 (abstain)
        
        Args:
            scores: Membership scores (m,)
        
        Returns:
            T: Guesses {-1, 0, +1}^m
        """
        # Initialize all to 0 (abstain)
        T = np.zeros(self.m, dtype=np.int32)
        
        # Get indices sorted by score (descending)
        sorted_indices = np.argsort(scores)[::-1]
        
        # Top k_plus → guess +1 (IN)
        top_k_indices = sorted_indices[:self.k_plus]
        T[top_k_indices] = +1
        
        # Bottom k_minus → guess -1 (OUT)
        bottom_k_indices = sorted_indices[-self.k_minus:]
        T[bottom_k_indices] = -1
        
        self.T = T
        
        n_abstain = self.m - self.k_plus - self.k_minus
        print(f"Guesses: +1={self.k_plus}, -1={self.k_minus}, "
              f"0={n_abstain}")
        
        return T
        # Converts scores to ternary guesses
        # Highest scores → +1, Lowest → -1, Middle → 0 (abstain)
    
    # ─── STEP 6: Statistical Analysis (Theorem 5.2) ────────────────────────
    def compute_audit_result(
        self,
        delta: float = 1e-5,
        confidence: float = 0.95
    ) -> Dict[str, Any]:
        """
        Algorithm 1, Line 9 + Theorem 5.2: Return S and T, then use
        Theorem 5.2 to compute ε lower bound.
        
        The test statistic is:
            W = Σ max{0, T_i · S_i}
        
        This counts correct guesses weighted by confidence.
        
        Args:
            delta: DP parameter δ
            confidence: Confidence level (e.g., 0.95 for 95%)
        
        Returns:
            Dictionary with audit results
        """
        assert self.S is not None and self.T is not None
        
        # Compute test statistic W = number of correct guesses
        W = 0
        for i in range(self.m):
            if self.T[i] * self.S[i] > 0:  # Same sign = correct guess
                W += abs(self.T[i])  # Weight by confidence (always 1 here)
        
        # Total number of guesses (excluding abstentions)
        r = self.k_plus + self.k_minus
        
        # Accuracy
        accuracy = W / r if r > 0 else 0
        
        print(f"\nAudit Results:")
        print(f"  Correct guesses: {W}/{r} ({accuracy:.2%})")
        
        # Compute ε lower bound using Theorem 5.2 (Corollary 5.4)
        eps_lower_bound = self._get_eps_lower_bound(
            m=self.m,
            r=r,
            v=W,
            delta=delta,
            confidence=confidence
        )
        
        results = {
            "m": self.m,
            "r": r,
            "correct_guesses": W,
            "accuracy": accuracy,
            "eps_lower_bound": eps_lower_bound,
            "delta": delta,
            "confidence": confidence,
            "S": self.S,
            "T": self.T,
            "scores": self.scores,
        }
        
        return results
        # Performs statistical test to get ε lower bound
        # Uses Corollary 5.4 from paper (binomial test)
        # Returns audit results with confidence interval
    
    # ─── Helper: ε Lower Bound (Corollary 5.4) ─────────────────────────────
    def _get_eps_lower_bound(
        self,
        m: int,
        r: int,
        v: int,
        delta: float,
        confidence: float
    ) -> float:
        """
        Implements Corollary 5.4 to compute ε lower bound.
        
        Uses binary search to find largest ε such that observing v
        correct guesses is statistically significant.
        
        Args:
            m: Number of canaries
            r: Number of guesses
            v: Number of correct guesses
            delta: DP parameter δ
            confidence: Confidence level
        
        Returns:
            eps_lower_bound: Lower bound on ε
        """
        p_value = 1 - confidence  # e.g., 0.05 for 95% confidence
        
        # Binary search for ε
        eps_min = 0.0
        eps_max = 10.0
        
        # Extend upper bound if needed
        while self._p_value_dp_audit(m, r, v, eps_max, delta) < p_value:
            eps_max *= 2
        
        # Binary search (30 iterations for precision)
        for _ in range(30):
            eps_mid = (eps_min + eps_max) / 2
            
            if self._p_value_dp_audit(m, r, v, eps_mid, delta) < p_value:
                eps_min = eps_mid
            else:
                eps_max = eps_mid
        
        return eps_min
        # Binary search to find tightest ε consistent with observations
        # Returns ε such that (ε, δ)-DP is rejected at given confidence
    
    def _p_value_dp_audit(
        self,
        m: int,
        r: int,
        v: int,
        eps: float,
        delta: float
    ) -> float:
        """
        Implements Corollary 5.4: p-value computation.
        
        If algorithm is (ε, δ)-DP, then correct guesses are bounded by
        Binomial(r, e^ε/(e^ε + 1)) + error term from δ.
        
        Args:
            m: Number of canaries
            r: Number of guesses
            v: Observed correct guesses
            eps: Hypothesized ε value
            delta: DP parameter δ
        
        Returns:
            p_value: Probability of observing ≥v correct guesses
        """
        # Accuracy of ε-DP randomized response
        q = 1 / (1 + np.exp(-eps))
        
        # β = P[Binomial(r, q) ≥ v]
        beta = stats.binom.sf(v - 1, r, q)
        
        # α correction term (handles δ)
        alpha = 0.0
        cumsum = 0.0
        
        for i in range(1, min(v + 1, m + 1)):
            cumsum += stats.binom.pmf(v - i, r, q)
            alpha = max(alpha, cumsum / i)
        
        # Final p-value (Corollary 5.4)
        p_value = beta + alpha * delta * 2 * m
        
        return min(p_value, 1.0)
        # Computes p-value for null hypothesis: algorithm is (ε, δ)-DP
        # Lower p-value → stronger evidence against (ε, δ)-DP


# ============================================================================
# COMPLETE ALGORITHM 1 EXECUTION PIPELINE
# ============================================================================

def run_algorithm1_audit(
    algorithm_A: Callable,      # DP algorithm to audit
    X_all: np.ndarray,          # All examples
    y_all: np.ndarray,          # All labels
    m: int,                     # Number of canaries
    k_plus: int,                # Positive guesses
    k_minus: int,               # Negative guesses
    loss_fn: Callable,          # Loss function
    delta: float = 1e-5,        # DP parameter δ
    confidence: float = 0.95,   # Confidence level
    seed: int = 42,
    **train_kwargs
) -> Dict[str, Any]:
    """
    Complete execution of Algorithm 1 from the paper.
    
    Pipeline:
        1. Generate random S ∈ {-1, +1}^m
        2. Partition data into x_IN and x_OUT
        3. Run algorithm A on x_IN → get w_0, w_T
        4. Compute scores Y_i = loss(w_0, x_i) - loss(w_T, x_i)
        5. Make guesses T based on scores
        6. Compute ε lower bound via Theorem 5.2
    
    Args:
        algorithm_A: Training function returning (w_0, w_T)
        X_all: All examples (n_total, ...)
        y_all: All labels (n_total,)
        m: Number of auditing examples (canaries)
        k_plus: Number of positive guesses
        k_minus: Number of negative guesses
        loss_fn: Loss function (model, x, y) -> loss
        delta: DP parameter δ
        confidence: Confidence level
        seed: Random seed
        **train_kwargs: Additional args for algorithm_A
    
    Returns:
        Dictionary with complete audit results
    """
    n_total = len(X_all)
    
    print("="*70)
    print("ALGORITHM 1: Privacy Auditor with One Training Run")
    print("="*70)
    print(f"Dataset size: {n_total}")
    print(f"Canaries (m): {m}")
    print(f"Guesses: k+ = {k_plus}, k- = {k_minus}")
    print(f"Target: (ε, δ = {delta})-DP with {confidence:.0%} confidence")
    print("="*70)
    
    # Initialize auditor
    auditor = PrivacyAuditor(
        m=m,
        k_plus=k_plus,
        k_minus=k_minus,
        n_total=n_total
    )
    
    # Step 1: Random selection
    print("\n[Step 1] Generating random inclusion/exclusion...")
    S = auditor.generate_random_selection(seed=seed)
    
    # Step 2: Partition dataset
    print("\n[Step 2] Partitioning dataset...")
    X_IN, y_IN, X_OUT, y_OUT = auditor.partition_dataset(X_all, y_all)
    
    # Step 3: Train algorithm A
    print("\n[Step 3] Training algorithm A...")
    w_0, w_T = auditor.run_training_algorithm(
        algorithm_A=algorithm_A,
        X_IN=X_IN,
        y_IN=y_IN,
        **train_kwargs
    )
    
    # Step 4: Compute scores
    print("\n[Step 4] Computing membership scores...")
    scores = auditor.compute_scores(
        X=X_all,
        y=y_all,
        loss_fn=loss_fn,
        model_w0=w_0,
        model_wT=w_T
    )
    
    # Step 5: Make guesses
    print("\n[Step 5] Making guesses based on scores...")
    T = auditor.make_guesses(scores)
    
    # Step 6: Statistical analysis
    print("\n[Step 6] Computing audit result (Theorem 5.2)...")
    results = auditor.compute_audit_result(delta=delta, confidence=confidence)
    
    print("\n" + "="*70)
    print("AUDIT COMPLETE")
    print("="*70)
    print(f"✓ Empirical ε lower bound: {results['eps_lower_bound']:.3f}")
    print(f"  (with {confidence:.0%} confidence, δ = {delta})")
    print("="*70)
    
    return results
    # End-to-end Algorithm 1 execution
    # Returns complete audit results with ε lower bound


# ============================================================================
# EXAMPLE USAGE
# ============================================================================

if __name__ == "__main__":
    print("Algorithm 1 implementation loaded successfully.")
    print("\nTo use:")
    print("  results = run_algorithm1_audit(")
    print("      algorithm_A=your_dp_training_function,")
    print("      X_all=your_data,")
    print("      y_all=your_labels,")
    print("      m=3000,")
    print("      k_plus=1500,")
    print("      k_minus=1500,")
    print("      loss_fn=your_loss_function")
    print("  )")

Algorithm 1 implementation loaded successfully.

To use:
  results = run_algorithm1_audit(
      algorithm_A=your_dp_training_function,
      X_all=your_data,
      y_all=your_labels,
      m=3000,
      k_plus=1500,
      k_minus=1500,
      loss_fn=your_loss_function
  )


In [2]:
# INTEGRATION GUIDE: Connect Algorithm 1 to Your Existing Code
# =============================================================

"""
This shows how to integrate algorithm1_standalone.py with your 
existing DP-SGD training code from the notebook.
"""

# ============================================================================
# STEP 1: Import the Algorithm 1 module
# ============================================================================

from algorithm1_standalone import run_algorithm1_audit
import torch
import torch.nn as nn


# ============================================================================
# STEP 2: Wrap Your Existing Training Code
# ============================================================================

def your_existing_dpsgd_wrapper(X_train, y_train, cfg, data_gen):
    """
    Wrapper around your train_shadow_model function.
    
    CRITICAL: Must return (w_0, w_T)
    - w_0 = model BEFORE any training
    - w_T = model AFTER all training
    """
    
    # 1. Build model
    model = build_model()  # Your WideResNet
    
    # 2. SAVE w_0 (initial weights) - THIS IS CRITICAL
    import copy
    model_w0 = copy.deepcopy(model)
    model_w0.eval()
    
    # 3. Train with DP-SGD (your existing code)
    model_wT = train_with_dpsgd(
        model=model,
        X_train=X_train,
        y_train=y_train,
        cfg=cfg
    )
    model_wT.eval()
    
    return model_w0, model_wT


# ============================================================================
# STEP 3: Define Loss Function
# ============================================================================

def loss_function(model, x, y):
    """
    Your loss function for scoring.
    
    Args:
        model: PyTorch model
        x: Batch of inputs (already tensors)
        y: Batch of labels (already tensors)
    
    Returns:
        scalar loss value
    """
    model.eval()
    with torch.no_grad():
        logits = model(x)
        loss = nn.CrossEntropyLoss()(logits, y)
    return loss


# ============================================================================
# STEP 4: Run the Audit
# ============================================================================

def run_audit_on_cifar10(cfg, data_gen):
    """
    Complete audit on your CIFAR-10 setup.
    """
    
    # Get your full dataset (with canary modifications)
    full_ds = data_gen.build_full_data()
    
    # Extract all data
    X_all = []
    y_all = []
    for i in range(len(full_ds)):
        x, y = full_ds[i]
        X_all.append(x.numpy())
        y_all.append(y.item())
    
    X_all = np.array(X_all)
    y_all = np.array(y_all)
    
    # Run Algorithm 1
    results = run_algorithm1_audit(
        algorithm_A=your_existing_dpsgd_wrapper,
        X_all=X_all,
        y_all=y_all,
        m=cfg['num_canaries'],          # 3000
        k_plus=cfg['k_plus'],            # 450
        k_minus=cfg['k_minus'],          # 450
        loss_fn=loss_function,
        delta=cfg['delta'],              # 1e-5
        confidence=0.95,
        seed=cfg['seed'],
        # Pass your config to wrapper
        cfg=cfg,
        data_gen=data_gen
    )
    
    return results


# ============================================================================
# USAGE IN YOUR NOTEBOOK
# ============================================================================

"""
In your Jupyter notebook, after loading the data and setting up CFG:

```python
# Import
from algorithm1_standalone import run_algorithm1_audit

# Run audit
results = run_audit_on_cifar10(CFG, data_gen)

# View results
print(f"ε lower bound: {results['eps_lower_bound']:.3f}")
print(f"Accuracy: {results['accuracy']:.2%}")
```
"""


# ============================================================================
# TROUBLESHOOTING
# ============================================================================

"""
Common Issues:

1. "w_0 not saved"
   → Make sure train_shadow_model returns BOTH w_0 and w_T
   → Add: w0_state = copy.deepcopy(model.state_dict()) BEFORE training

2. "GPU OOM during scoring"
   → Move models to CPU before scoring
   → Process in smaller batches

3. "Scores all similar"
   → Check that w_0 and w_T are actually different
   → Verify loss function is working correctly

4. "Low accuracy"
   → This is expected for strong DP! 
   → Try weaker noise (lower noise_multiplier) to verify audit works
"""

ModuleNotFoundError: No module named 'algorithm1_standalone'